# 02 - Preprocesamiento de datos

Este cuaderno aplica las decisiones de preprocesamiento definidas durante el análisis exploratorio.

El objetivo principal es crear un conjunto de datos limpio para la tarea de clasificación binaria de la orientación política.

Pasos del preprocesamiento:

- Cargar las variables seleccionadas.
- Eliminar los valores perdidos.
- Eliminar las categorías de «sin respuesta» en P5328.
- Eliminar las categorías ideológicas centrales.
- Transformar P5328 en una variable objetivo binaria.
- Eliminar la variable P5328 original para evitar la fuga de datos.
- Guardar el conjunto de datos procesado.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_DATA_PATH = Path("../data/raw/empalme_politica.csv")
PROCESSED_DATA_PATH = Path("../data/processed/political_orientation_clean.csv")

PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Cargar datos

In [10]:
data = pd.read_csv(RAW_DATA_PATH)

print("Original dataset shape:", data.shape)

data.head()

Original dataset shape: (64759, 420)


,DIRECTORIO,NRO_ENCUESTA,HOGAR_NUMERO,PERSONA_NUMERO,P606,P2057,P2059,P2061,P2063,P2063S3,...,P2051S2,P2051S3,P2051S4,P2051S5,P2051S6,P2051S7,P2051S8,P2051S9,P2051S10,P2051S11
0,112214,252,1,1,6.0,2.0,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,112214,252,1,2,4.0,2.0,2.0,2.0,1.0,NaN,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,112214,252,1,3,1.0,2.0,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,112214,252,1,4,1.0,2.0,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,112214,252,1,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
columnas = [
    "P220",
    "P2057",
    "P605",
    "P6210",
    "P6945",
    "P3039",
    "P2001S1",
    "P2001S2",
    "P2001S3",
    "P2001S4",
    "P2001S17",
    "P2001S7",
    "P2001S12",
    "P2001S13",
    "P2001S14",
    "P2001S15",
    "P2001S18",
    "P2003S4",
    "P2001S10",
    "P2003S17",
    "P2003S7",
    "P2003S10",
    "P2003S12",
    "P2003S13",
    "P2003S14",
    "P2003S15",
    "P2003S18",
    "P5373S4",
    "P5373S5",
    "P5373S6",
    "P5306S1",
    "P5306S2",
    "P2019",
    "P2016S1",
    "P2016S2",
    "P2016S3",
    "P2016S4",
    "P2016S5",
    "P2016S6",
    "P2016S7",
    "P2016S8",
    "P2016S9",
    "P2017S1",
    "P2017S2",
    "P2017S3",
    "P2017S4",
    "P2017S5",
    "P2017S6",
    "P2021",
    "P1754",
    "P5376S1",
    "P5376S2",
    "P5376S3",
    "P5376S4",
    "P5376S5",
    "P5376S6",
    "P5317S4",
    "P5317S5",
    "P5307S1",
    "P5307S2",
    "P5307S3",
    "P5307S4",
    "P5307S5",
    "P2017S7",
    "P5314S2",
    "P5314S3",
    "P5314S4",
    "P5314S5",
    "P5314S6",
    "P5314S7",
    "P5317S1",
    "P5317S9",
    "P2009S9",
    "P5328"
]

In [11]:
missing_columns = [col for col in columnas if col not in data.columns]

if missing_columns:
    print("Missing columns:")
    print(missing_columns)
else:
    print("All selected columns are available.")

All selected columns are available.


In [12]:
df = data[columnas].copy()

print("Selected dataset shape:", df.shape)

df.head()

Selected dataset shape: (64759, 74)


,P220,P2057,P605,P6210,P6945,P3039,P2001S1,P2001S2,P2001S3,P2001S4,...,P5314S2,P5314S3,P5314S4,P5314S5,P5314S6,P5314S7,P5317S1,P5317S9,P2009S9,P5328
0,1,2.0,3.0,6.0,1.0,1.0,2.0,2.0,2.0,2.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0
1,2,2.0,3.0,5.0,1.0,2.0,2.0,2.0,2.0,2.0,...,1.0,1.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,3.0
2,2,2.0,2.0,6.0,1.0,2.0,2.0,2.0,2.0,2.0,...,1.0,1.0,2.0,2.0,2.0,2.0,2.0,1.0,1.0,5.0
3,1,2.0,1.0,5.0,1.0,1.0,2.0,2.0,2.0,2.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0
4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Eliminar valores faltantes

In [15]:
initial_rows = len(df)

df = df.dropna()

final_rows = len(df)
removed_rows = initial_rows - final_rows
removed_percentage = (removed_rows / initial_rows) * 100

print("Initial rows:", initial_rows)
print("Rows after removing missing values:", final_rows)
print("Removed rows:", removed_rows)
print(f"Removed percentage: {removed_percentage:.2f}%") 

Initial rows: 64759
Rows after removing missing values: 46392
Removed rows: 18367
Removed percentage: 28.36%


In [16]:
df["P5328"].value_counts().sort_index()

P5328
1.0      1981
2.0       688
3.0      1660
4.0      1906
5.0     16878
6.0      2723
7.0      1894
8.0      1989
9.0      1010
10.0     5255
98.0     5865
99.0     4543
Name: count, dtype: int64

## 3. Eliminar respuestas no válidas

In [17]:
df = df[
    ~df["P5328"].isin([98, 99])
].copy()

print("Shape after removing 98 and 99:", df.shape)

df["P5328"].value_counts().sort_index()

Shape after removing 98 and 99: (35984, 74)


P5328
1.0      1981
2.0       688
3.0      1660
4.0      1906
5.0     16878
6.0      2723
7.0      1894
8.0      1989
9.0      1010
10.0     5255
Name: count, dtype: int64

## 4. Eliminar categorías centrales

In [18]:
df = df[
    ~df["P5328"].isin([5, 6])
].copy()

print("Shape after removing central categories 5 and 6:", df.shape)

df["P5328"].value_counts().sort_index()

Shape after removing central categories 5 and 6: (16383, 74)


P5328
1.0     1981
2.0      688
3.0     1660
4.0     1906
7.0     1894
8.0     1989
9.0     1010
10.0    5255
Name: count, dtype: int64

## 5. Crear variable objetivo binaria

In [19]:
df["political_orientation"] = np.where(
    df["P5328"] <= 4,
    0,
    1
)

df["political_orientation"].value_counts().sort_index()

political_orientation
0     6235
1    10148
Name: count, dtype: int64

In [20]:
class_distribution = (
    df["political_orientation"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

class_distribution

political_orientation
0    38.057743
1    61.942257
Name: proportion, dtype: float64

Las decisiones sobre el balance de las clases se toman en el entrenamiento del modelo

## 6. Eliminar P5328 para evitar fuga de datos

In [21]:
df = df.drop(columns=["P5328"])

df.head()

,P220,P2057,P605,P6210,P6945,P3039,P2001S1,P2001S2,P2001S3,P2001S4,...,P5314S2,P5314S3,P5314S4,P5314S5,P5314S6,P5314S7,P5317S1,P5317S9,P2009S9,political_orientation
0,1,2.0,3.0,6.0,1.0,1.0,2.0,2.0,2.0,2.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0
1,2,2.0,3.0,5.0,1.0,2.0,2.0,2.0,2.0,2.0,...,1.0,1.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,0
7,2,2.0,1.0,5.0,5.0,2.0,2.0,2.0,2.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0
10,2,1.0,2.0,5.0,1.0,2.0,2.0,2.0,2.0,2.0,...,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,0
11,1,1.0,2.0,3.0,1.0,1.0,2.0,2.0,2.0,2.0,...,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,0


In [22]:
print("Final processed dataset shape:", df.shape)

df.info()

Final processed dataset shape: (16383, 74)
<class 'pandas.core.frame.DataFrame'>
Index: 16383 entries, 0 to 64757
Data columns (total 74 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   P220                   16383 non-null  int64  
 1   P2057                  16383 non-null  float64
 2   P605                   16383 non-null  float64
 3   P6210                  16383 non-null  float64
 4   P6945                  16383 non-null  float64
 5   P3039                  16383 non-null  float64
 6   P2001S1                16383 non-null  float64
 7   P2001S2                16383 non-null  float64
 8   P2001S3                16383 non-null  float64
 9   P2001S4                16383 non-null  float64
 10  P2001S17               16383 non-null  float64
 11  P2001S7                16383 non-null  float64
 12  P2001S12               16383 non-null  float64
 13  P2001S13               16383 non-null  float64
 14  P2001S14        

In [23]:
df.isna().sum().sum()

np.int64(0)

## 7. Dataset procesado

In [24]:
df.to_csv(PROCESSED_DATA_PATH, index=False)

print("Processed dataset saved at:", PROCESSED_DATA_PATH)

Processed dataset saved at: ..\data\processed\political_orientation_clean.csv


In [25]:
processed_df = pd.read_csv(PROCESSED_DATA_PATH)

print("Loaded processed dataset shape:", processed_df.shape)

processed_df.head()

Loaded processed dataset shape: (16383, 74)


,P220,P2057,P605,P6210,P6945,P3039,P2001S1,P2001S2,P2001S3,P2001S4,...,P5314S2,P5314S3,P5314S4,P5314S5,P5314S6,P5314S7,P5317S1,P5317S9,P2009S9,political_orientation
0,1,2.0,3.0,6.0,1.0,1.0,2.0,2.0,2.0,2.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0
1,2,2.0,3.0,5.0,1.0,2.0,2.0,2.0,2.0,2.0,...,1.0,1.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,0
2,2,2.0,1.0,5.0,5.0,2.0,2.0,2.0,2.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0
3,2,1.0,2.0,5.0,1.0,2.0,2.0,2.0,2.0,2.0,...,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,0
4,1,1.0,2.0,3.0,1.0,1.0,2.0,2.0,2.0,2.0,...,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,0
